In [ ]:
from ugot import ugot

got = ugot.UGOT() 
got.initialize("192.168.1.118")
got.load_models(["word_recognition"])

print("Done!")

192.168.1.211:50051
Done!


In [11]:
got.get_words_result()

'Hello then'

In [ ]:
try:
    while True:
        # Read the text the robot sees
        text = got.get_words_result()

        # Print it out on the display screen
        got.screen_print_text_newline(text=text, color=1)
        got.play_audio_tts(data=text, voice_type=0, wait=True)

        if text in {"Forward"}:
            got.mecanum_move_speed_times(direction=0, speed=30, times=50, unit=1)
        elif text in {"Left"}:
            got.mecanum_turn_speed_times(turn=2, speed=60, times=90, unit=2)
        elif text in {"Right"}:
            got.mecanum_turn_speed_times(turn=3, speed=60, times=90, unit=2)
        elif text in {"Backward"}:
            got.mecanum_move_speed_times(direction=1, speed=30, times=50, unit=1)

        got.screen_clear()
except KeyboardInterrupt:
    print("Stopped")

In [ ]:
try:
    while True:
        # Read the text the robot sees
        text = got.get_words_result()

        if not text:
            continue

        # Print it out on the display screen
        got.screen_print_text_newline(text=text, color=1)
        got.play_audio_tts(data=text, voice_type=0, wait=True)

        # Split "Forward 50" into ["Forward", "50"]
        parts = text.strip().split()
        command = parts[0].capitalize()

        # Use the number if there is one, otherwise use None
        amount = None
        if len(parts) > 1 and parts[1].isdigit():
            amount = int(parts[1])

        if command == "Forward":
            got.mecanum_move_speed_times(direction=0, speed=30, times=amount or 50, unit=1)
        elif command == "Backward":
            got.mecanum_move_speed_times(direction=1, speed=30, times=amount or 50, unit=1)
        elif command == "Left":
            got.mecanum_turn_speed_times(turn=2, speed=60, times=amount or 90, unit=2)
        elif command == "Right":
            got.mecanum_turn_speed_times(turn=3, speed=60, times=amount or 90, unit=2)
        
        got.screen_clear()
except KeyboardInterrupt:
    got.mecanum_stop()
    print("Stopped!")

In [ ]:
MAX_DISTANCE = 100   # cm
MAX_ANGLE = 360      # degrees

# command: (function type, direction/turn value, speed, default amount)
COMMANDS = {
    "Forward":  ("move", 0, 30, 50),
    "Backward": ("move", 1, 30, 50),
    "Left":     ("turn", 2, 60, 90),
    "Right":    ("turn", 3, 60, 90),
}

last_text = None

try:
    while True:
        text = got.get_words_result()

        # Nothing seen: reset so the same card can be used again later
        if not text:
            last_text = None
            continue

        # Same card still in view: don't repeat it
        if text == last_text:
            continue
        last_text = text

        got.screen_print_text_newline(text=text, color=1)
        got.play_audio_tts(data=text, voice_type=0, wait=True)

        parts = text.strip().split()
        command = parts[0].capitalize()

        if command not in COMMANDS:
            got.play_audio_tts(data="Unknown command", voice_type=0, wait=True)
            got.screen_clear()
            continue

        kind, value, speed, default = COMMANDS[command]

        amount = default
        if len(parts) > 1 and parts[1].isdigit():
            amount = int(parts[1])

        if kind == "move":
            amount = min(amount, MAX_DISTANCE)
            got.mecanum_move_speed_times(direction=value, speed=speed, times=amount, unit=1)
        else:
            amount = min(amount, MAX_ANGLE)
            got.mecanum_turn_speed_times(turn=value, speed=speed, times=amount, unit=2)

        got.screen_clear()
except KeyboardInterrupt:
    print("Stopped")